In [40]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [41]:
products = pd.read_csv("../Dataset/Products.csv")
suppliers = pd.read_csv("../Dataset/Suppliers.csv")
warehouses = pd.read_csv("../Dataset/Warehouses.csv")
regions = pd.read_csv("../Dataset/Regions.csv")
inventory = pd.read_csv("../Dataset/Inventory.csv")

In [42]:
print("Products:", products.shape)
print("Suppliers:", suppliers.shape)
print("Warehouses:", warehouses.shape)
print("Regions:", regions.shape)
print("Inventory:", inventory.shape)

Products: (25, 6)
Suppliers: (10, 6)
Warehouses: (5, 6)
Regions: (5, 2)
Inventory: (25, 6)


In [43]:
start_date = datetime(2023, 1, 1)
end_date = datetime(2025, 12, 31)

date_range = pd.date_range(start=start_date, end=end_date)

print("Total Days:", len(date_range))

Total Days: 1096


In [44]:
region_weights = {
    "R001": 0.22,  # North
    "R002": 0.18,  # South
    "R003": 0.18,  # East
    "R004": 0.17,  # West
    "R005": 0.25   # Central
}

In [45]:
def seasonal_multiplier(category, month):

    multiplier = 1.0

    if category == "Electronics" and month in [11, 12]:
        multiplier = 2.0

    elif category == "Home Appliances" and month in [4, 5, 6]:
        multiplier = 1.7

    elif category == "Furniture" and month in [6, 7, 8]:
        multiplier = 1.5

    elif category == "Personal Care" and month in [2, 11, 12]:
        multiplier = 1.6

    return multiplier

In [46]:
print(seasonal_multiplier("Electronics", 12))
print(seasonal_multiplier("Furniture", 7))
print(seasonal_multiplier("Office Supplies", 5))

2.0
1.5
1.0


In [47]:
orders = []

In [48]:
print(products.columns.tolist())
print(suppliers.columns.tolist())
print(warehouses.columns.tolist())
print(regions.columns.tolist())
print(inventory.columns.tolist())

['Product_ID', 'Product_Name', 'Category', 'Unit_Cost', 'Unit_Price', 'Demand_Type']
['Supplier_ID', 'Supplier_Name', 'Country', 'Reliability_Score', 'Lead_Time_Days', 'Supplier_Type']
['Warehouse_ID', 'Warehourse_Name', 'Region', 'Capacity', 'Utilization', 'Warehouse_Type']
['Region_ID', 'Region_Name']
['Inventory_ID', 'Product_ID', 'Warehouse_ID', 'Stocl_Level', 'Reorder_Point', 'Inventory_Status']


In [49]:
num_orders = 20000

for i in range(1, num_orders + 1):

    order_id = f"O{i:06d}"

    order_date = random.choice(date_range)

    product = products.sample(1).iloc[0]

    product_id = product["Product_ID"]

    category = product["Category"]

    month = order_date.month

    multiplier = seasonal_multiplier(category, month)

    quantity = max(
        1,
        int(
            np.random.poisson(
                3 * multiplier
            )
        )
    )

    region_id = random.choices(
        population=list(region_weights.keys()),
        weights=list(region_weights.values())
    )[0]

    warehouse = warehouses.sample(1).iloc[0]
    warehouse_id = warehouse["Warehouse_ID"]

    supplier = suppliers.sample(1).iloc[0]
    supplier_id = supplier["Supplier_ID"]

    revenue = quantity * product["Unit_Price"]

    fulfillment_status = random.choices(
        ["Fulfilled", "Delayed", "Pending"],
        weights=[90, 7, 3]
    )[0]

    orders.append([
        order_id,
        order_date,
        product_id,
        supplier_id,
        warehouse_id,
        region_id,
        quantity,
        revenue,
        fulfillment_status
    ])

In [50]:
orders_df = pd.DataFrame(
    orders,
    columns=[
        "Order_ID",
        "Order_Date",
        "Product_ID",
        "Supplier_ID",
        "Warehouse_ID",
        "Region_ID",
        "Quantity",
        "Revenue",
        "Fulfillment_Status"
    ]
)

In [51]:
orders_df.head()

,Order_ID,Order_Date,Product_ID,Supplier_ID,Warehouse_ID,Region_ID,Quantity,Revenue,Fulfillment_Status
0,O000001,2025-01-05,P007,S009,W002,R005,6,1074,Fulfilled
1,O000002,2025-08-05,P008,S004,W004,R002,4,996,Fulfilled
2,O000003,2025-08-31,P010,S008,W001,R003,2,458,Fulfilled
3,O000004,2023-10-24,P022,S006,W002,R004,1,79,Fulfilled
4,O000005,2024-03-15,P004,S005,W005,R004,2,498,Fulfilled


In [52]:
orders_df.shape

(20000, 9)

In [53]:
orders_df["Revenue"].sum()

np.int64(19512866)

In [54]:
orders_df["Fulfillment_Status"].value_counts(normalize=True) * 100

Fulfillment_Status
Fulfilled    89.77
Delayed       7.13
Pending       3.10
Name: proportion, dtype: float64

In [55]:
orders_df.to_csv(
    "../Dataset/Orders.csv",
    index=False
)

print("Orders.csv Created Successfully")

Orders.csv Created Successfully


In [56]:
orders_df.head()
orders_df.shape
orders_df["Revenue"].sum()

np.int64(19512866)

In [57]:
orders_df.shape

(20000, 9)

In [58]:
shipments = []

In [59]:
transport_modes = [
    "Air",
    "Road",
    "Rail",
    "Express Courier",
    "Standard Shipping"
]

for index, row in orders_df.iterrows():

    shipment_id = f"SHP{index+1:06d}"

    order_date = pd.to_datetime(row["Order_Date"])

    transport_mode = random.choice(transport_modes)

    if transport_mode == "Air":
        delivery_days = random.randint(1, 3)
        shipping_cost = random.randint(80, 150)

    elif transport_mode == "Express Courier":
        delivery_days = random.randint(2, 4)
        shipping_cost = random.randint(40, 90)

    elif transport_mode == "Road":
        delivery_days = random.randint(3, 7)
        shipping_cost = random.randint(20, 60)

    elif transport_mode == "Rail":
        delivery_days = random.randint(5, 10)
        shipping_cost = random.randint(15, 45)

    else:
        delivery_days = random.randint(7, 14)
        shipping_cost = random.randint(10, 35)

    ship_date = order_date + timedelta(days=random.randint(0, 2))

    delivery_date = ship_date + timedelta(days=delivery_days)

    delivery_status = random.choices(
        ["On-Time", "Delayed"],
        weights=[88, 12]
    )[0]

    shipments.append([
        shipment_id,
        row["Order_ID"],
        transport_mode,
        shipping_cost,
        ship_date,
        delivery_date,
        delivery_status
    ])

In [60]:
shipments_df = pd.DataFrame(
    shipments,
    columns=[
        "Shipment_ID",
        "Order_ID",
        "Transport_Mode",
        "Shipping_Cost",
        "Ship_Date",
        "Delivery_Date",
        "Delivery_Status"
    ]
)

In [61]:
shipments_df.head()

,Shipment_ID,Order_ID,Transport_Mode,Shipping_Cost,Ship_Date,Delivery_Date,Delivery_Status
0,SHP000001,O000001,Express Courier,79,2025-01-07,2025-01-10,On-Time
1,SHP000002,O000002,Air,106,2025-08-05,2025-08-06,On-Time
2,SHP000003,O000003,Express Courier,63,2025-08-31,2025-09-02,On-Time
3,SHP000004,O000004,Air,116,2023-10-25,2023-10-26,On-Time
4,SHP000005,O000005,Rail,18,2024-03-16,2024-03-23,Delayed


In [62]:
shipments_df.shape

(20000, 7)

In [63]:
shipments_df.to_csv(
    "../Dataset/Shipments.csv",
    index=False
)

print("Shipments.csv Created Successfully")

Shipments.csv Created Successfully


In [5]:
import pandas as pd

orders_df = pd.read_csv("../Dataset/Orders.csv")

In [6]:
print("Total Rows:", len(orders_df))

print("Unique Order IDs:", orders_df["Order_ID"].nunique())

print("Duplicate IDs:", orders_df["Order_ID"].duplicated().sum())

Total Rows: 20000
Unique Order IDs: 20000
Duplicate IDs: 0


In [2]:
import pandas as pd

orders_df = pd.read_csv("../Dataset/Orders.csv")

In [3]:
orders_df["Order_Date"] = pd.to_datetime(
    orders_df["Order_Date"],
    dayfirst=True
)

In [4]:
orders_df["Order_Date"] = orders_df["Order_Date"].dt.strftime("%Y-%m-%d")

In [5]:
orders_df.to_csv(
    "../Dataset/Orders.csv",
    index=False
)

print("Date format fixed")

Date format fixed


In [6]:
orders_df.head()

,Order_ID,Order_Date,Product_ID,Supplier_ID,Warehouse_ID,Region_ID,Quantity,Revenue,Fulfillment_Status
0,O000001,2025-01-05,P007,S009,W002,R005,6,1074,Fulfilled
1,O000002,2025-08-05,P008,S004,W004,R002,4,996,Fulfilled
2,O000003,2025-08-31,P010,S008,W001,R003,2,458,Fulfilled
3,O000004,2023-10-24,P022,S006,W002,R004,1,79,Fulfilled
4,O000005,2024-03-15,P004,S005,W005,R004,2,498,Fulfilled


In [6]:
import pandas as pd

shipments_df = pd.read_csv("../Dataset/Shipments.csv")
shipments_df.head()
shipments_df.head()

,Shipment_ID,Order_ID,Transport_Mode,Shipping_Cost,Ship_Date,Delivery_Date,Delivery_Status
0,SHP000001,O000001,Express Courier,79,07-01-2025,10-01-2025,On-Time
1,SHP000002,O000002,Air,106,05-08-2025,06-08-2025,On-Time
2,SHP000003,O000003,Express Courier,63,31-08-2025,02-09-2025,On-Time
3,SHP000004,O000004,Air,116,25-10-2023,26-10-2023,On-Time
4,SHP000005,O000005,Rail,18,16-03-2024,23-03-2024,Delayed


In [7]:
shipments_df["Ship_Date"] = pd.to_datetime(
    shipments_df["Ship_Date"],
    dayfirst=True
).dt.strftime("%Y-%m-%d")

shipments_df["Delivery_Date"] = pd.to_datetime(
    shipments_df["Delivery_Date"],
    dayfirst=True
).dt.strftime("%Y-%m-%d")

In [8]:
shipments_df.to_csv(
    "../Dataset/Shipments.csv",
    index=False
)